# Electronic Sales Analysis

Optional exploratory view over the tested `electronics_sales` package. Run the command-line pipeline for reproducible artifacts; use this notebook to inspect tables and charts.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

PROJECT_ROOT = next((p for p in (Path.cwd(), Path.cwd().parent) if (p / 'pyproject.toml').is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Open this notebook from the cloned project directory.')
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from electronics_sales import build_metrics, clean_orders, customer_summary
sns.set_theme(style='whitegrid')

In [ ]:
CSV_PATH = PROJECT_ROOT / 'data' / 'raw' / 'Electronic_sales_Sep2023-Sep2024.csv'
if not CSV_PATH.is_file():
    raise FileNotFoundError(f'Download the documented source CSV and save it at {CSV_PATH}.')

raw = pd.read_csv(CSV_PATH, dtype={'Customer ID': 'string'})
orders = clean_orders(raw)
customers = customer_summary(orders)
metrics = build_metrics(orders)
display(pd.Series(orders.attrs['cleaning_report'], name='rows').to_frame())
display(customers.head(10))

## Reconciled metric tables

Zeros in a cohort matrix are observed months without orders; blank cells are future months outside the observation window.

In [ ]:
for name, table in metrics.items():
    print(name)
    display(table)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=metrics['revenue_by_shipping'], x='shipping_method', y='revenue', ax=axes[0], color='#3478A3')
sns.barplot(data=metrics['revenue_by_product'], x='product_type', y='revenue', ax=axes[1], color='#278579')
axes[0].set_title('Revenue by shipping method')
axes[1].set_title('Revenue by product type')
for ax in axes:
    ax.tick_params(axis='x', rotation=30)
fig.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
sns.heatmap(metrics['cohort_revenue'] / 1_000, annot=True, fmt='.1f', cmap='Blues', ax=ax)
ax.set(title='Cohort revenue, thousands of units', xlabel='Months since first order', ylabel='First-order month')
fig.tight_layout()